# Практика №1. Логистическая регрессия

**Задача:** бинарная классификация банкнот по числовым признакам, извлечённым из изображений. Целевая переменная UCI — `class`.

**Датасет:** UCI Banknote Authentication, 1 372 наблюдения и 4 числовых признака. Источник: [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/267/banknote-authentication), DOI: `10.24432/C55P57`.

Логистическая регрессия оценивает вероятность принадлежности объекта к классу. В этой работе отдельно проверяется качество на validation и test.

In [ ]:
%pip install -q ucimlrepo pandas numpy matplotlib seaborn scikit-learn joblib

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from ucimlrepo import fetch_ucirepo

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

## 1. Загрузка данных

Загружаем признаки и целевой класс из UCI. Перед моделированием проверим фактические значения целевой переменной и размеры таблицы.

In [ ]:
dataset = fetch_ucirepo(id=267)
X_raw = dataset.data.features.copy()
y_raw = dataset.data.targets.iloc[:, 0].copy()

print(f"Размер матрицы признаков: {X_raw.shape}")
print("Целевая переменная:", dataset.data.targets.columns[0])
print("Классы:", sorted(pd.Series(y_raw).dropna().unique().tolist()))
display(X_raw.head())
display(pd.Series(y_raw).value_counts().sort_index().to_frame("count"))

## 2. Проверка качества и очистка

Ищем пропуски, бесконечные значения, дубликаты и статистические выбросы по IQR. В признаках нет универсальных физических границ, поэтому значения за пределами IQR отмечаются для анализа, но не удаляются без обоснования. Пропуски признаков при необходимости будут заполнены медианой внутри Pipeline.

In [ ]:
df = X_raw.copy()
df["target"] = pd.to_numeric(y_raw, errors="coerce")
df = df.replace([np.inf, -np.inf], np.nan)

print("Пропуски до очистки:")
display(df.isna().sum().to_frame("missing"))
print("Количество полных дубликатов:", int(df.duplicated().sum()))

rows_before = len(df)
df = df.dropna(subset=["target"]).drop_duplicates().reset_index(drop=True)
print(f"Удалено строк с отсутствующей целевой переменной/дубликатов: {rows_before - len(df)}")
print("Пропуски в признаках после проверки:")
display(df.drop(columns="target").isna().sum().to_frame("missing"))

X = df.drop(columns="target")
y = df["target"].astype(int)

outlier_counts = {}
for column in X.columns:
    q1, q3 = X[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    outlier_counts[column] = int(((X[column] < q1 - 1.5 * iqr) | (X[column] > q3 + 1.5 * iqr)).sum())
print("Потенциальные выбросы по IQR (не удаляются автоматически):")
display(pd.Series(outlier_counts, name="iqr_outliers").to_frame())
display(df.describe().T)

## 3. Разведочный анализ данных (EDA)

Посмотрим баланс классов, распределения четырёх признаков и корреляционную матрицу. Поскольку признаки числовые, для корреляций используется исходная шкала измерений.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for column, ax in zip(X.columns, axes.ravel()):
    for class_value in sorted(df["target"].unique()):
        class_data = df.loc[df["target"] == class_value, column]
        ax.hist(
            class_data, bins=25, density=True, histtype="step",
            linewidth=1.5, label=f"class {class_value}",
        )
    ax.set_title(f"Распределение: {column}")
    ax.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
class_counts = df["target"].value_counts().sort_index()
plt.bar(class_counts.index.astype(str), class_counts.values)
plt.title("Распределение целевых классов")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Корреляционная матрица")
plt.tight_layout()
plt.show()

## 4. Разделение выборки и масштабирование

Выполняем стратифицированное разделение на train/validation/test (примерно 70/15/15), чтобы сохранить соотношение классов. Медианная импутация и стандартизация выполняются внутри Pipeline и обучаются только на соответствующей обучающей части.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)
print(f"Train: {len(X_train)}, validation: {len(X_val)}, test: {len(X_test)}")
print("Доли классов в test:")
display(y_test.value_counts(normalize=True).sort_index().to_frame("share"))

numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

## 5. Выбор параметра C на validation

Параметр `C` регулирует силу регуляризации логистической регрессии. Сравним несколько значений по Macro F1 на валидационной выборке. Тестовая выборка на этом этапе не используется.

In [ ]:
validation_results = []
best_c = None
best_val_f1 = -1.0

for c_value in [0.1, 1.0, 10.0]:
    candidate = Pipeline([
        ("preprocessor", numeric_preprocessor),
        ("model", LogisticRegression(C=c_value, max_iter=2000, random_state=RANDOM_STATE)),
    ])
    candidate.fit(X_train, y_train)
    prediction = candidate.predict(X_val)
    score = f1_score(y_val, prediction, average="macro")
    validation_results.append({"C": c_value, "validation_macro_f1": score})
    if score > best_val_f1:
        best_val_f1 = score
        best_c = c_value

display(pd.DataFrame(validation_results).sort_values("validation_macro_f1", ascending=False))
print(f"Выбрано C={best_c}; validation Macro F1={best_val_f1:.4f}")

## 6. Финальное обучение и оценка на test

После выбора `C` объединяем train и validation, обучаем окончательную модель и только один раз оцениваем её на test. Помимо Accuracy приводим Balanced Accuracy, Precision, Recall, F1 и ROC-AUC.

In [ ]:
X_train_full = pd.concat([X_train, X_val], axis=0)
y_train_full = pd.concat([y_train, y_val], axis=0)

final_model = Pipeline([
    ("preprocessor", numeric_preprocessor),
    ("model", LogisticRegression(C=best_c, max_iter=2000, random_state=RANDOM_STATE)),
])
final_model.fit(X_train_full, y_train_full)
y_test_pred = final_model.predict(X_test)
y_test_proba = final_model.predict_proba(X_test)[:, list(final_model.classes_).index(1)]

test_metrics = {
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_test_pred),
    "Precision (class 1)": precision_score(y_test, y_test_pred, pos_label=1, zero_division=0),
    "Recall (class 1)": recall_score(y_test, y_test_pred, pos_label=1, zero_division=0),
    "F1 (class 1)": f1_score(y_test, y_test_pred, pos_label=1, zero_division=0),
    "Macro F1": f1_score(y_test, y_test_pred, average="macro"),
    "ROC-AUC": roc_auc_score(y_test, y_test_proba),
}
display(pd.Series(test_metrics).to_frame("value"))
print(classification_report(y_test, y_test_pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred, cmap="Blues")
plt.title("Матрица ошибок: Logistic Regression")
plt.tight_layout()
plt.show()

## 7. Сохранение модели и вывод

Сохраняем весь Pipeline, включая импутацию и масштабирование: при последующем использовании к новым объектам будет применяться та же предобработка. В итоговом отчёте укажите выбранное значение `C` и метрики тестовой выборки.

In [ ]:
artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(final_model, artifact_dir / "logistic_regression_banknote.joblib")
with (artifact_dir / "logistic_regression_metrics.json").open("w", encoding="utf-8") as file:
    json.dump({"best_C": best_c, "validation_macro_f1": best_val_f1, "test": test_metrics}, file, ensure_ascii=False, indent=2)
print("Модель и метрики сохранены в папку artifacts/")